<img src="https://raw.githubusercontent.com/pasdptt/public_teaching_tools4DS/main/assets/brand/logo-160.png" width="80" align="right" alt="Course logo">

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pasdptt/public_teaching_tools4DS/blob/main/modules/m02-evaluation-leakage/lab/lab2_evaluation_leakage.ipynb)

# Lab 2 — Evaluation and leakage

In [ ]:
# Setup: run this cell first (Google Colab or a local Jupyter kernel).
COURSE_REF = "main"  # version of the course repository to install the helper package from
try:
    import tsds
except ImportError:  # first run in a fresh Colab runtime: install the course helper package
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    f"tsds @ git+https://github.com/pasdptt/public_teaching_tools4DS@{COURSE_REF}#subdirectory=src"], check=True)
    import tsds

tsds.env.check()          # prints versions; warns if a package is outside the tested range
SEED = tsds.seed_everything()
rng = tsds.rng()

import sys
import numpy as np
import pandas as pd
from sklearn.model_selection import (train_test_split, KFold, GroupKFold, StratifiedKFold,
                                     RepeatedStratifiedKFold, cross_val_score)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.feature_selection import SelectKBest, f_classif
try:
    import imblearn
except ImportError:  # not preinstalled in this runtime
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "imbalanced-learn"], check=True)
from imblearn.pipeline import make_pipeline as imb_make_pipeline
from imblearn.over_sampling import SMOTE

**Course:** 040563107 Tools for Statistical Data Science · **Module m02** · **Time:** 3 hours · **Graded lab G1** (3⅓% of the course)

**After this lab you can:**
* compare a model with a baseline using a confidence interval for test accuracy (objective m02-O2);
* choose a validation scheme that matches the data's structure, here grouped data (O1);
* measure leakage on pure noise and explain why its size depends on whether the step uses $y$ (O3, O5);
* treat resampling for class imbalance as a fitted operation, inside cross-validation (O6);
* find and remove target leakage in an AI-written analysis (O4, AI-audit task).

**Two steps:** in the lab session we **run through** the ideas together. Then, as homework, you **do this lab on your own**
from a fresh copy and upload the `.ipynb` to the G1 assignment in Microsoft Teams. **That homework is what is marked.**

**How it is marked** ([lab rubric](https://github.com/pasdptt/public_teaching_tools4DS/blob/main/course/rubrics.md#lab-rubric-graded-labs)):
your notebook is re-run from top to bottom in a fresh session and checked with hidden tests (correctness 3 points,
reproducibility 1 point); staff mark your written answers (failure analysis 2, interpretation 2).

**How to work:**
1. Run the setup cell first.
2. Complete every cell marked **Task**. Replace `raise NotImplementedError()` with your code, then run the **Self-check** cell below it; it prints `OK` when your answer passes. Do not edit given cells or self-check cells.
3. Write the answers marked **Q** after *Your answer:* (at most 100 words each).
4. Before submitting, use *Runtime → Restart session and run all*. The notebook must run top to bottom without errors.
5. Fill in the **AI-use log** at the end.

## Part 1 — A baseline and an interval (about 30 minutes)

Digits, recoded as a yes/no task: **is the digit a 9?** Only about 10% of the images are 9s, so a model that always says
"not 9" is already about 90% accurate. A test accuracy means little until it is compared with that baseline, with its uncertainty.

The **Wilson interval** for a proportion $\hat p = k/n$ (here: $k$ correct predictions out of $n$ test cases) is

$$\frac{\hat p + \frac{z^2}{2n}}{1 + \frac{z^2}{n}} \;\pm\; \frac{z}{1 + \frac{z^2}{n}}\sqrt{\frac{\hat p(1-\hat p)}{n} + \frac{z^2}{4n^2}},$$

with $z = 1.96$ for 95%. Unlike the textbook interval $\hat p \pm z\sqrt{\hat p(1-\hat p)/n}$, it stays inside $[0, 1]$ and
behaves well when $\hat p$ is near 0 or 1.

In [ ]:
Xd, yd = tsds.data.load("digits")
Xd = Xd.to_numpy()
y9 = (yd.to_numpy() == 9).astype(int)
print(f"{y9.sum()} nines out of {len(y9)} images ({y9.mean():.1%})")
X_tr, X_te, y_tr, y_te = train_test_split(Xd, y9, test_size=0.25, stratify=y9, random_state=SEED)
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))

In [ ]:
# Task T1
# (a) wilson_ci(k, n, z=1.96): return (lower, upper) of the Wilson interval for k successes out of n.
# (b) base_acc: test accuracy of DummyClassifier(strategy="most_frequent") fitted on the training set.
#     test_acc: test accuracy of `model` fitted on the training set.
# (c) ci_base, ci_model: Wilson intervals for base_acc and test_acc (k = number of correct test predictions).
def wilson_ci(k, n, z=1.96):
    # YOUR CODE HERE
    raise NotImplementedError()


# YOUR CODE HERE
raise NotImplementedError()
print(f"baseline {base_acc:.3f}  95% CI {np.round(ci_base, 3)}")
print(f"model    {test_acc:.3f}  95% CI {np.round(ci_model, 3)}")

In [ ]:
# Self-check for task T1: run it; it should print OK.
lo, hi = wilson_ci(0, 10)
assert np.isclose(lo, 0, atol=1e-12) and 0.2 < hi < 0.35, "k = 0 should give a lower limit of 0 and a positive upper limit"
lo, hi = wilson_ci(7, 20)
assert lo < 7 / 20 < hi and np.isclose(lo + hi, 2 * (7 / 20 + 1.96**2 / 40) / (1 + 1.96**2 / 20)), "check the centre of the interval"
assert np.isclose(base_acc, 1 - y_te.mean()), "the baseline always predicts the majority class"
assert ci_model[0] <= test_acc <= ci_model[1] and ci_base[0] <= base_acc <= ci_base[1]
print("OK")

**Q1.** **Interpretation.** Is the model better than the baseline? Answer with both intervals, and say what the comparison would look like
if the test set had only 40 images instead of 450.

*Your answer:* 

## Part 2 — Grouped data (about 30 minutes)

40 patients, 10 measurements each. Measurements of the same patient are similar (a patient-specific centre), and the
outcome is a property of the **patient**, unrelated to the features in any way that would carry over to a new patient.
A model that will be used on **new patients** must be validated on patients it has not seen.

In [ ]:
gen = np.random.default_rng(SEED)
n_patients, per_patient = 40, 10
patient_label = gen.integers(0, 2, n_patients)
patient_centre = gen.normal(0, 3, size=(n_patients, 5))
Xg = np.repeat(patient_centre, per_patient, axis=0) + gen.normal(0, 0.5, size=(n_patients * per_patient, 5))
yg = np.repeat(patient_label, per_patient)
groups = np.repeat(np.arange(n_patients), per_patient)
knn = KNeighborsClassifier(n_neighbors=1)
print(Xg.shape, "patients:", len(np.unique(groups)))

In [ ]:
# Task T2
# (a) grouped_cv_score(est, X, y, groups): mean accuracy of `est` over GroupKFold(n_splits=5),
#     so that no group appears in both a training and a test fold.
# (b) kfold_acc: mean accuracy of `knn` with KFold(n_splits=5, shuffle=True, random_state=SEED) (rows split at random).
#     group_acc: grouped_cv_score(knn, Xg, yg, groups).
def grouped_cv_score(est, X, y, groups):
    # YOUR CODE HERE
    raise NotImplementedError()


# YOUR CODE HERE
raise NotImplementedError()
print(f"KFold {kfold_acc:.3f} | GroupKFold {group_acc:.3f}")

In [ ]:
# Self-check for task T2: run it; it should print OK.
assert kfold_acc > 0.95, "with rows split at random, 1-NN finds the same patient in the training folds"
assert group_acc < kfold_acc - 0.2, "with patients kept together, the score should drop sharply"
print("OK")

**Q2.** **Interpretation.** Why is the KFold score so high, and which number would you report for a model used on new patients? Name one
dataset you might use in your project where rows are grouped, and what the groups are.

*Your answer:* 

## Part 3 — How large is leakage? (about 30 minutes)

Pure noise: $n = 100$ cases, $p = 1000$ features, random labels. **Any** honest estimate of accuracy must be about 0.5.
We compare two fitted steps, each done once **before** cross-validation on all the data, or **inside** a `Pipeline` so that it
is refitted on each training fold:
* `SelectKBest(f_classif, k=20)` chooses features **using $y$**;
* `StandardScaler()` estimates means and standard deviations **without $y$**.

In [ ]:
gen = np.random.default_rng(SEED)
Xn = gen.normal(size=(100, 1000))
yn = gen.integers(0, 2, size=100)
rcv = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=SEED)
clf = LogisticRegression(max_iter=5000)


def repeated_cv(est, X, y):
    return cross_val_score(est, X, y, cv=rcv).mean()

In [ ]:
# Task T3
# Use repeated_cv and clf. Four numbers:
#   skb_before: SelectKBest(f_classif, k=20) fitted on ALL of (Xn, yn), then CV of clf on the selected features
#   skb_inside: CV of make_pipeline(SelectKBest(f_classif, k=20), clf) on (Xn, yn)
#   sc_before:  StandardScaler fitted on ALL of Xn, then CV of clf
#   sc_inside:  CV of make_pipeline(StandardScaler(), clf)
# YOUR CODE HERE
raise NotImplementedError()
print(pd.Series({"select before": skb_before, "select inside": skb_inside,
                 "scale before": sc_before, "scale inside": sc_inside}).round(3))

In [ ]:
# Self-check for task T3: run it; it should print OK.
assert skb_before > skb_inside + 0.2, "supervised selection before the split should look much better than it is"
assert abs(sc_before - sc_inside) < 0.05, "an unsupervised scaler fitted before the split should barely matter here"
assert abs(skb_inside - 0.5) < 0.15 and abs(sc_inside - 0.5) < 0.1, "inside the pipeline, noise should score near 0.5"
print("OK")

**Q3.** **Interpretation.** Explain, using your four numbers, why selecting features before cross-validation inflates the estimate so much
while scaling before cross-validation barely changes it. Is "fit inside the pipeline" still the right rule for the scaler? Why?

*Your answer:* 

## Part 4 — Class imbalance: resampling is a fitted step (about 35 minutes)

Back to "is it a 9?" (10% positives). We compare ways of dealing with imbalance, all with the **same** 5-fold stratified split
and the **same** metric, the F1 score of the positive class (`scoring="f1"`):

* plain logistic regression;
* `class_weight="balanced"`;
* SMOTE (synthetic oversampling of the minority class) **inside** an `imblearn` pipeline, so it is applied to training folds only;
* SMOTE applied to the **whole** dataset first, then cross-validation (a common mistake).

`imblearn.pipeline.make_pipeline` (imported as `imb_make_pipeline`) is needed because scikit-learn's own pipeline cannot hold a resampler.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

In [ ]:
# Task T4
# Fill the dict f1 with mean CV F1 scores (cv=cv, scoring="f1") on (Xd, y9) of
#   "plain":        make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))
#   "weighted":     the same with LogisticRegression(max_iter=5000, class_weight="balanced")
#   "smote_inside": imb_make_pipeline(StandardScaler(), SMOTE(random_state=SEED), LogisticRegression(max_iter=5000))
#   "smote_before": SMOTE(random_state=SEED).fit_resample on ALL of (Xd, y9), then CV of the "plain" pipeline
# YOUR CODE HERE
raise NotImplementedError()
print(pd.Series(f1).round(3))

In [ ]:
# Self-check for task T4: run it; it should print OK.
assert set(f1) == {"plain", "weighted", "smote_inside", "smote_before"}
assert all(0 < v <= 1 for v in f1.values())
assert f1["smote_before"] > f1["smote_inside"] + 0.05, "SMOTE before the split should look better than it is"
print("OK")

**Q4.** **Failure analysis.** Give two separate reasons why the `smote_before` score cannot be trusted. Then say, from your numbers, whether
either remedy for imbalance actually improved F1 over the plain model, and what you would report.

*Your answer:* 

## Part 5 — AI audit: an analysis that is too good (about 35 minutes)

A clinic records five variables for 600 patients and whether they were later diagnosed with a disease. The goal is a model that
predicts the diagnosis **at the first visit**, from what is known then. An AI assistant was asked to "build and evaluate a
classifier for `disease`", and wrote the cell below. **Do not edit it.**

Column meanings: `age` (years), `blood_pressure` (mm Hg), `cholesterol` (mg/dL), `smoker` (0/1), `treatment_started`
(1 if treatment for the disease was started; recorded by the clinic after the diagnosis), `disease` (the outcome).

In [ ]:
gen = np.random.default_rng(SEED + 5)
n = 600
age = gen.normal(55, 12, n)
bp = gen.normal(130, 15, n)
chol = gen.normal(200, 30, n)
smoker = gen.integers(0, 2, n)
risk = 0.04 * (age - 55) + 0.03 * (bp - 130) + 0.01 * (chol - 200) + 0.8 * smoker - 0.4
disease = (gen.random(n) < 1 / (1 + np.exp(-risk))).astype(int)
treatment_started = np.where(gen.random(n) < 0.93, disease, 1 - disease)
clinic = pd.DataFrame({"age": age.round(0), "blood_pressure": bp.round(0), "cholesterol": chol.round(0),
                       "smoker": smoker, "treatment_started": treatment_started, "disease": disease})
clinic.head()

In [ ]:
# ---- AI-generated cell (do not edit) ----
X_all = clinic.drop(columns="disease")
y_all = clinic["disease"]
ai_model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))
ai_acc = cross_val_score(ai_model, X_all, y_all, cv=cv).mean()
print(f"Cross-validated accuracy: {ai_acc:.3f}. The model is ready for use at the first visit.")

In [ ]:
# Task T5
# (a) leak_col: the name of the column that makes the AI's estimate invalid for the stated use.
# (b) honest_acc: mean CV accuracy (cv=cv) of the same pipeline without that column.
# (c) majority_acc: accuracy of always predicting the most common class (the baseline).
# YOUR CODE HERE
raise NotImplementedError()
print(f"AI {ai_acc:.3f} | honest {honest_acc:.3f} | majority baseline {majority_acc:.3f}")

In [ ]:
# Self-check for task T5: run it; it should print OK.
assert leak_col in clinic.columns and leak_col != "disease"
assert honest_acc < ai_acc - 0.15, "removing the leaking column should lower the estimate substantially"
assert honest_acc > majority_acc, "the honest model should still beat the baseline"
assert np.isclose(majority_acc, max(clinic["disease"].mean(), 1 - clinic["disease"].mean()))
print("OK")

**Q5.** **Failure analysis (AI audit).** What kind of leakage is this, how did you find it, and what question should be asked of every
feature before it is used? Would a better validation scheme (more folds, a test set) have caught it?

*Your answer:* 

## AI-use log (required; see the course AI policy)

| Tool | What you asked or used it for | What you accepted or changed | How you verified it |
|---|---|---|---|
| *e.g. Colab AI* | *the imblearn pipeline syntax* | *used it; set random_state myself* | *T4 self-check passed; read the imblearn docs* |
| | | | |

If you used no AI tool, write "None".